In [12]:
import pandas as pd 
import nflreadpy as nfl

#built in python library that makes building file paths easier
from pathlib import Path


In [13]:
#
SEASONS = list(range(2020, 2027))   # 2020 through 2026

RAW = Path("../data/raw")   #notebook is in notebooks folder, so need to go up one level to get to data folder

RAW.mkdir(parents=True, exist_ok=True) # create the folder if missing; do nothing if it exists



In [14]:
# one row per player per week, all 7 seasons stacked
# nflreadpy returns a Polars table; convert it to pandas
stats = nfl.load_player_stats(SEASONS, summary_level="week").to_pandas()
stats.shape

(119897, 150)

In [15]:
# Groups the stats by season then, then looks at the week column and gets the min, max, and count of weeks for each season. 
# This is useful to see how many weeks of data we have for each season and the range of weeks.
print(stats.groupby("season")["week"].agg(["min", "max", "count"]))

# how many single-week player stats we have for each season type (regular season, postseason)
print(stats["season_type"].value_counts())

print(stats.columns)  # see all columns in the stats dataframe

#selects a subset of columns from the stats dataframe and displays the first 5 rows. 
# This is useful to see what the data looks like and what columns are available for analysis.
# Double square brackets are used to select multiple columns from the dataframe.
stats[["player_id", "player_display_name", "position", "season", "week", "fantasy_points_ppr"]].head()


        min  max  count
season                 
2020      1   21  17602
2021      1   22  18969
2022      1   22  18831
2023      1   22  18643
2024      1   22  18983
2025      1   22  19422
2026      1    7   7447
season_type
REG     114806
POST      5091
Name: count, dtype: int64
Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'season', 'week', 'season_type',
       'game_id',
       ...
       'pt_out_of_bounds', 'pt_downed', 'pt_touchback', 'pt_fair_caught',
       'pt_returned', 'pt_return_yards', 'pt_return_tds', 'pt_net_yards',
       'fantasy_points', 'fantasy_points_ppr'],
      dtype='str', length=150)


,player_id,player_display_name,position,season,week,fantasy_points_ppr
0,00-0019596,Tom Brady,QB,2020,1,20.46
1,00-0020531,Drew Brees,QB,2020,1,14.40
2,00-0022127,Jason Witten,TE,2020,1,1.20
3,00-0022824,Andy Lee,P,2020,1,0.00
4,00-0022921,Larry Fitzgerald,WR,2020,1,7.40


In [16]:
stats.to_parquet(RAW / "player_stats_week_2020_2026.parquet", index=False)

In [17]:
schedules = nfl.load_schedules(SEASONS)  # one row per game, 2020-2026 (nflverse keeps all seasons in one file)
schedules = schedules.to_pandas()        # convert from Polars to pandas
schedules.shape                          # (rows, columns): quick check that data came back

(1965, 46)

In [18]:
print(schedules.groupby("season")["game_id"].count())  # games per season
print(schedules["game_type"].value_counts())            # REG = regular season; WC/DIV/CON/SB = playoff rounds
print(schedules.columns)  # see all columns in the schedules dataframe
schedules[["game_id", "season", "week", "home_team", "away_team",  # show key columns:
           "total_line", "spread_line", "home_rest", "away_rest"]].tail()  # Vegas lines and rest days

season
2020    269
2021    285
2022    284
2023    285
2024    285
2025    285
2026    272
Name: game_id, dtype: int64
game_type
REG    1887
WC       36
DIV      24
CON      12
SB        6
Name: count, dtype: int64
Index(['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday',
       'gametime', 'away_team', 'away_score', 'home_team', 'home_score',
       'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis',
       'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest',
       'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds',
       'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game',
       'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id',
       'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee',
       'stadium_id', 'stadium'],
      dtype='str')


,game_id,season,week,home_team,away_team,total_line,spread_line,home_rest,away_rest
1960,2026_18_CHI_MIN,2026,18,MIN,CHI,NaN,NaN,7,7
1961,2026_18_MIA_NE,2026,18,NE,MIA,NaN,NaN,7,7
1962,2026_18_TB_NO,2026,18,NO,TB,NaN,NaN,7,7
1963,2026_18_PHI_NYG,2026,18,NYG,PHI,NaN,NaN,7,7
1964,2026_18_DAL_WAS,2026,18,WAS,DAL,NaN,NaN,7,7


In [19]:
snaps = nfl.load_snap_counts(SEASONS)    # one row per player per game: how many snaps he played
snaps = snaps.to_pandas()                # convert from Polars to pandas
snaps.shape                              # (rows, columns): quick check

(167597, 16)

In [20]:
print(snaps.groupby("season")["game_id"].count())       # # Count the number of snap data records (rows) for each NFL season
print(snaps["position"].value_counts().head(15))         # includes defense and special teams; you'll filter later : essentially, this is a sanity check to see if the data looks reasonable and if there are any unexpected values in the position column, and snaps per position
print(snaps["offense_pct"].describe())                   # is snap % stored as 0-1 or 0-100? note which (answer): stored as 0-1, so you can multiply by 100 to get % later
print(snaps["pfr_player_id"].isna().sum())               # Count snap-count records missing a PFR player ID, which may prevent matching players when merging dataset
snaps[["pfr_player_id", "player", "position", "season",  # show key columns:
       "week", "offense_snaps", "offense_pct"]].head()   # ID, name, when, and snap usage

season
2020    24999
2021    26468
2022    26381
2023    26540
2024    26615
2025    26613
2026     9981
Name: game_id, dtype: int64
position
LB    23876
WR    18585
CB    16826
TE    11197
RB    11048
DE    10779
T     10287
DT     9559
G      8947
C      4780
FS     4765
QB     4503
SS     4477
S      4466
P      3600
Name: count, dtype: int64
count    167597.000000
mean          0.236287
std           0.368251
min           0.000000
25%           0.000000
50%           0.000000
75%           0.420000
max           1.000000
Name: offense_pct, dtype: float64
0


,pfr_player_id,player,position,season,week,offense_snaps,offense_pct
0,TomlLa01,Laken Tomlinson,G,2020,1,62.0,1.0
1,BrunDa00,Daniel Brunskill,T,2020,1,62.0,1.0
2,McGlMi00,Mike McGlinchey,T,2020,1,62.0,1.0
3,WillTr21,Trent Williams,T,2020,1,62.0,1.0
4,GrasHr00,Hroniss Grasu,C,2020,1,62.0,1.0


In [21]:
players = nfl.load_players()                             # one row per player ever, with IDs from many sites
players = players.to_pandas()                            # convert from Polars to pandas
ids = players[["gsis_id", "pfr_id"]].dropna()            # keep only the two IDs we need; drop players missing either
print(f"{len(ids):,} players have both a GSIS and a PFR ID")  # how many can be matched

22,698 players have both a GSIS and a PFR ID


In [22]:
# Save each table to data/raw/ so later notebooks can load it instead of re-downloading.
# RAW / "file.parquet" joins the folder and file name into one path (/ means "join" for Path objects).
# index=False stops pandas from saving its row numbers (0, 1, 2...) as an extra column.

schedules.to_parquet(RAW / "schedules_2020_2026.parquet", index=False)  # games: teams, dates, scores, Vegas lines, rest days
snaps.to_parquet(RAW / "snap_counts_2020_2026.parquet", index=False)    # player-games: snaps played, keyed by PFR ID
ids.to_parquet(RAW / "player_ids.parquet", index=False)                 # translator: GSIS ID <-> PFR ID for each player

# Confirm the saves: find every .parquet file in data/raw/ and print just the file names, sorted A-Z
print("Saved:", sorted(p.name for p in RAW.glob("*.parquet")))

Saved: ['player_ids.parquet', 'player_stats_week_2020_2026.parquet', 'schedules_2020_2026.parquet', 'snap_counts_2020_2026.parquet']
